### Environment Setup (Colab) & Hugging Face Authentication

In [ ]:
!pip install -q bitsandbytes accelerate torchao>=0.16.0
!pip install -q "pytorch-lightning>=1.8.0,<2.0.0"
!pip install -q transformers datasets peft evaluate sacrebleu unbabel-comet huggingface_hub

from huggingface_hub import notebook_login

print("[*] Log in to your Hugging Face account to enable model uploading:")
notebook_login()

### Preprocessing Pipeline

In [ ]:
# Uncomment commented code areas if you want to preprocess your dataset in token format.

import os
import re
import unicodedata
import pandas as pd
# import torch
# from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

# NLLB_MODEL_ID = "facebook/nllb-200-distilled-600M"

# tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL_ID, src_lang="eng_Latn", tgt_lang="kik_Latn")    # Change to appropriate target language code, for this notebook it's strictly languages that were originally trained on base NLLB model, here we are using Kikuyu. Check NLLB base model to know supported languages.

def load_and_preprocess_pipeline(
    csv_name,
    # tokenizer,
    src_col,
    tgt_col,
    max_src_len,
    max_tgt_len,
    min_src_len,
    min_tgt_len,
    drop_on_truncation,
    seed
):
    # assert tokenizer is not None, "Pass `tokenizer`"

    home_dir = os.path.expanduser("~")
    path_options = [
        os.path.join(home_dir, "Downloads", csv_name),
        os.path.join("/content", "Downloads", csv_name),
        os.path.join(os.getcwd(), csv_name),
    ]

    download_path = None
    for p in path_options:
        if os.path.exists(p):
            download_path = p
            print(f"[*] Found dataset at: {download_path}")
            break
    if download_path is None:
        raise FileNotFoundError(f"Could not find '{csv_name}' in any expected locations: {path_options}.")

    # Load CSV
    df = pd.read_csv(download_path)
    if src_col not in df.columns or tgt_col not in df.columns:
        # assume first two columns are src/tgt if file has no header
        df = pd.read_csv(download_path, header=None, names=[src_col, tgt_col])

    # Drop missing
    df = df.dropna(subset=[src_col, tgt_col]).copy()

    # Clean
    def clean_text(x):
        x = unicodedata.normalize("NFC", str(x))
        x = re.sub(r"\s+", " ", x).strip()
        return x

    df[src_col] = df[src_col].apply(clean_text)
    df[tgt_col] = df[tgt_col].apply(clean_text)

    # Remove identical pairs
    df = df[
        df[src_col].str.strip().str.lower() != df[tgt_col].str.strip().str.lower()
    ].copy()

    # Remove exact duplicates of the full pair
    df = df.drop_duplicates(subset=[src_col, tgt_col]).reset_index(drop=True)

    # Shuffle to avoid any ordering bias before splitting
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    # src_texts = df[src_col].tolist()
    # tgt_texts = df[tgt_col].tolist()

    # # Source lengths uncapped
    # src_enc = tokenizer(
    #     src_texts,
    #     truncation=False,
    #     padding=False,
    #     add_special_tokens=True,
    # )
    # # Target lengths uncapped
    # tgt_enc = tokenizer(
    #     tgt_texts,
    #     truncation=False,
    #     padding=False,
    #     add_special_tokens=True,
    # )

    # df["en_tok_len"] = [len(x) for x in src_enc["input_ids"]]
    # df["kik_tok_len"] = [len(x) for x in tgt_enc["input_ids"]]

    # # Basic min filtering
    # df = df[(df["en_tok_len"] >= min_src_len) & (df["kik_tok_len"] >= min_tgt_len)].copy()

    # if drop_on_truncation:
    #     df = df[(df["en_tok_len"] <= max_src_len) & (df["kik_tok_len"] <= max_tgt_len)].copy()

    # Final columns
    df = df[[src_col, tgt_col]].reset_index(drop=True)

    raw_dataset = Dataset.from_pandas(df, preserve_index=False)

    # 90/5/5 split (train/validation/test)
    train_test = raw_dataset.train_test_split(test_size=0.1, seed=seed, shuffle=True)
    test_valid = train_test["test"].train_test_split(test_size=0.5, seed=seed, shuffle=True)

    final_dataset = DatasetDict({
        "train": train_test["train"],
        "validation": test_valid["train"],
        "test": test_valid["test"],
    })

    print(f"[✓] Data preprocessing complete. Rows kept: {len(df)}")
    print(f"[✓] Split overview:\n{final_dataset}")
    return final_dataset

dataset_package = load_and_preprocess_pipeline(
    "kikuyu_dataset.csv",
    # tokenizer=tokenizer,
    src_col="english",
    tgt_col="kikuyu",
    max_src_len=512,
    max_tgt_len=512,
    min_src_len=2,
    min_tgt_len=2,
    drop_on_truncation=True,      # drop examples that hit the max boundary
    seed=42
)

#####Parameter-Efficient Fine-Tuning (PEFT) using Low-Rank Adaptation (LoRA)

In [ ]:
!pip install -q torchao>=0.16.0
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType,prepare_model_for_kbit_training

NLLB_MODEL_ID = "facebook/nllb-200-distilled-600M"
HF_USERNAME = "Huggingface-username"
HUB_REPO_NAME = "Repository-name"
FULL_REPO_ID = f"{HF_USERNAME}/{HUB_REPO_NAME}"

# Initialize tokenizer with target prefix setups
tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL_ID, src_lang="eng_Latn", tgt_lang="kik_Latn")

# 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    NLLB_MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    tie_word_embeddings=False
)

model = prepare_model_for_kbit_training(model)

# Tokenization sequence mapper
def tokenize_nllb_fn(examples):
    inputs = examples['english']
    targets = examples['kikuyu']
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding=False)
    labels = tokenizer(targets, max_length=512, truncation=True, padding=False)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Vector map
tokenized_datasets = dataset_package.map(tokenize_nllb_fn, batched=True, remove_columns=['english', 'kikuyu'])

# Inject Parameter-Efficient PEFT LoRA matrices configuration
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Data collator pads tensors inside individual batches
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_translation_peft",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=True,
    logging_steps=50,
    report_to="none",
    push_to_hub=True,
    hub_model_id=FULL_REPO_ID,
    hub_strategy="every_save",               # "default", "every_save", "end", "checkpoint"
    hub_private_repo=True
)

# Initialize object mapping configurations
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

# Start finetuning
print("[*] Starting training loop...")
trainer.train()

print(f"[*] Post-training check: Pushing final Kikuyu weights and tokenizer structure to {FULL_REPO_ID}...")
trainer.push_to_hub(commit_message="Training complete.")
print("[✓] Model successfully live on Hugging Face Hub!")

### Evaluation

In [ ]:
!pip install -q evaluate sacrebleu
import evaluate
import sacrebleu
import torch

print("[*] Launching NLLB Evaluator Generation Matrix Loop across target validation splits...")

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

eval_subset = dataset_package["test"].select(range(min(len(dataset_package["test"]), 200)))

references = []
predictions = []

model.eval()
# Ensure correct target language code
target_lang_id = tokenizer.convert_tokens_to_ids("kik_Latn")

for item in eval_subset:
    inputs = tokenizer(item['english'], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=target_lang_id,
            max_length=512
        )
    decoded_pred = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
    predictions.append(decoded_pred)
    references.append(item['kikuyu'])

bleu_results = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
chrf_results = chrf_metric.compute(predictions=predictions, references=[[r] for r in references])

print("\n================ FINAL NLLB PERFORMANCE REPORT ================")
print(f"| -> SacreBLEU Score  : {bleu_results['score']:.3f}")
print(f"| -> chrF++ Score     : {chrf_results['score']:.3f}")
print("===============================================================\n")

In [ ]:
import pandas as pd

original_english = [item['english'] for item in eval_subset]

analysis_df = pd.DataFrame({
    "English (Source)": original_english,
    "Ground Truth (Reference)": references,
    "Adapter Output (Prediction)": predictions
})

analysis_df['Perfect Match'] = analysis_df['Ground Truth (Reference)'].str.strip() == analysis_df['Adapter Output (Prediction)'].str.strip()

print(f"Showing sample translations from the test split (Total Perfect Matches: {analysis_df['Perfect Match'].sum()}/{len(analysis_df)}):")
display(analysis_df.head(20))

In [ ]:
def translate_new_sentence(english_text, model, tokenizer, target_lang="kik_Latn"):
    """
    Translates completely unseen English sentence.
    """
    model.eval()

    inputs = tokenizer(english_text, return_tensors="pt").to(model.device)

    target_lang_id = tokenizer.convert_tokens_to_ids(target_lang)

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=target_lang_id,
            max_length=512,
            num_beams=5,
            early_stopping=True
        )

    translated_text = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
    return translated_text

print("Translations ---")
print("Type 'quit' or 'exit' to close\n")

while True:
    user_sentence = input("Enter an English sentence to translate: ")

    if user_sentence.lower().strip() in ['quit', 'exit']:
        print("Closed.")
        break

    if not user_sentence.strip():
        print("Please enter a valid sentence.")
        print("-" * 40)
        continue

    translation = translate_new_sentence(user_sentence, model, tokenizer, target_lang="kik_Latn")

    print(f"Translation: {translation}")
    print("-" * 40)